<a href="https://colab.research.google.com/github/sponfork/Saki/blob/main/Tensor_converter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-cache-dir --force-reinstall --no-deps xformers trl peft accelerate bitsandbytes datasets

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-7coq2svy/unsloth_e4a700fd58bd46a19d78ed1d90327604
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-7coq2svy/unsloth_e4a700fd58bd46a19d78ed1d90327604
  Resolved https://github.com/unslothai/unsloth.git to commit 13ef43494732fbec65b1263ed4a8d0cb4b51a98c
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 278.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 301.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 318.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 135.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 285.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 452.2 MB/s  0:00:00
  Created wheel for 

In [3]:
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from trl import SFTTrainer
from transformers import TrainingArguments

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0,
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",
)

dataset = load_dataset("json", data_files="saki_training_dataset.jsonl", split="train")

def format_prompts(batch):
    texts = [
        tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False)
        for msg in batch["messages"]
    ]
    return {"text": texts}

formatted_dataset = dataset.map(format_prompts, batched=True)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        output_dir="saki_7b_output",
        optim="adamw_8bit",
    ),
)

trainer.train()

==((====))==  Unsloth 2026.11.4: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.11.4 patched 28 layers with 28 QKV layers, 28 O layers and 0 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/113 [00:00<?, ? examples/s]

... (2 outputs left out)

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,3.871978
10,3.240046
15,2.598478
20,2.000891
25,1.445641
30,1.042106
35,0.892700
40,0.765411
45,0.644626
50,0.550857


Unsloth: Restored added_tokens_decoder metadata in saki_7b_output/checkpoint-60/tokenizer_config.json.


TrainOutput(global_step=60, training_loss=1.502601158618927, metrics={'train_runtime': 406.7635, 'train_samples_per_second': 1.18, 'train_steps_per_second': 0.148, 'total_flos': 4178412930908160.0, 'train_loss': 1.502601158618927, 'epoch': 4.0})

In [4]:
model.save_pretrained("saki_7b_mlbb_adapter")
tokenizer.save_pretrained("saki_7b_mlbb_adapter")
print("[✔] LoRA adapter saved to saki_7b_mlbb_adapter folder!")


Unsloth: Restored added_tokens_decoder metadata in saki_7b_mlbb_adapter/tokenizer_config.json.


[✔] LoRA adapter saved to saki_7b_mlbb_adapter folder!


In [5]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

messages = [
    {
        "role": "system",
        "content": "You are Saki, an elite Mobile Legends: Bang Bang (MLBB) draft strategy coach. Provide sharp, analytical counter-pick advice based on professional match data."
    },
    {
        "role": "user",
        "content": "The enemy team just picked Fanny and Joy for their core side. What heroes should we draft to counter them?"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    temperature=0.7,
    use_cache=True,
)

response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("=== SAKI'S DRAFT ADVICE ===")
print(response)

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== SAKI'S DRAFT ADVICE ===
By squeezing out high flexibility and utility, draft around heroes like Valir, Atlas, or Ares. Their ability to disassemble key threats around Fanny and Joy will secure your win.


In [7]:
import json

with open("/content/mlbb_heroes_clean.json", "r", encoding="utf-8") as f:
    heroes_db = json.load(f)

print(f"[✔] Loaded {len(heroes_db)} heroes into heroes_db successfully!")

[✔] Loaded 126 heroes into heroes_db successfully!


In [8]:
import json
import gradio as gr

def generate_saki_advice(enemy_picks):
    if not enemy_picks:
        return "Select at least one enemy hero to generate draft counter analysis."

    matched_data = []
    for hero in heroes_db:
        if hero["name"] in enemy_picks:
            matched_data.append(hero)

    context_data = json.dumps(matched_data, indent=2)

    user_prompt = f"The enemy team locked in: {', '.join(enemy_picks)}. What are our best counter picks and strategy?"

    messages = [
        {
            "role": "system",
            "content": (
                "You are Saki, an elite Mobile Legends: Bang Bang (MLBB) draft strategy coach. "
                "Provide sharp, analytical counter-pick advice using strictly the statistical data context provided.\n\n"
                f"STATISTICAL DATA CONTEXT:\n{context_data}"
            ),
        },
        {"role": "user", "content": user_prompt},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=300,
        temperature=0.7,
        use_cache=True,
    )

    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

all_hero_names = sorted([hero["name"] for hero in heroes_db])

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🏆 Saki: MLBB Live Ranked Draft Coach")
    gr.Markdown("Simulate an active draft room. Pick enemy heroes to get real-time counter advice from Saki.")

    with gr.Row():
        with gr.Column(scale=2):
            enemy_selection = gr.CheckboxGroup(
                choices=all_hero_names,
                label="🔴 Enemy Team Draft (Select Locked Heroes)",
                interactive=True
            )
            submit_btn = gr.Button("🧠 Ask Saki for Counter Advice", variant="primary")

        with gr.Column(scale=1):
            saki_output = gr.Textbox(
                label="💬 Saki's Tactical Analysis",
                lines=12,
                interactive=False
            )

    submit_btn.click(
        fn=generate_saki_advice,
        inputs=[enemy_selection],
        outputs=[saki_output]
    )

demo.launch(inline=False, share=True)

/tmp/ipykernel_590/2645613260.py:51: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f53cf32fc9329e6c84.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
